# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook guides you through loading, exploring, and analyzing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All references to core dataset components (record sets, fields) use their Croissant `@id`s for precise and schema-consistent handling.

### Dataset Source
The dataset source Croissant schema URL is:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

**Note:** Data covers ordered logistic regression outputs and survey-based records on rangeland management in Northern Kenya.

In [ ]:
# Install mlcroissant if it's not already present
!pip install --quiet mlcroissant

## 1. Data Loading

First, load the metadata for the dataset and display its high-level summary. We use the Croissant schema URL and let `mlcroissant` handle the loading.

*You may see a warning about package installation if running in a new environment.*

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # this is an object, not a dict
# Display dataset title and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review all available record sets and their fields, with all items referenced by their `@id`. This provides the structure you can use for specific data extractions later.

**Note:** The dataset may contain several record sets (tabular or entity groupings); for analysis, you need to know their `@id`s and the fields under each.

In [ ]:
# Find all record sets (`@id` and human-readable name if available)
# Each record set in the dataset.metadata.record_sets property (list of RecordSet objects)

if hasattr(metadata, 'record_sets') and metadata.record_sets:
    print("Available Record Sets:")
    for rs in metadata.record_sets:
        print(f"- @id: {rs.id} | name: {getattr(rs, 'name', 'N/A')}")

    # For each record set, print its fields
    print("\nRecord Set Fields:")
    for rs in metadata.record_sets:
        print(f"\nRecord Set @id: {rs.id}")
        if hasattr(rs, 'fields'):
            for f in rs.fields:
                print(f"  - Field @id: {f.id} | name: {getattr(f, 'name', 'N/A')}")
        else:
            print("  No fields found.")
else:
    print("No record sets discovered in this dataset. Check dataset.metadata.record_sets property.")

## 3. Data Extraction
Load records from a selected record set into a Pandas DataFrame for analysis.

- Use the `@id` of the record set and its fields, as identified above.
- If there are multiple record sets, loop over all and load them separately. 
- Dataframes are keyed using the record set `@id`.

In [ ]:
# Collect all record set @id's for extraction

# Get all record sets (if any)
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_set_ids = [rs.id for rs in metadata.record_sets]
    print(f"Extracting record sets: {record_set_ids}")
else:
    record_set_ids = []

dataframes = {}
for rs_id in record_set_ids:
    # Each record yields a dictionary
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nLoaded {len(df)} records from record set {rs_id} with columns:")
    print(df.columns.tolist())

# Example: Show first 5 rows of the first record set, if present
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nSample rows for record set {first_rs_id}:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Perform basic processing steps (filtering, normalization, and grouping) on a chosen numeric field in one record set.

1. **Numeric fields:** Identify a numeric field (`@id`) from the data overview.
2. **Filtering:** Remove records where numeric values are below a threshold.
3. **Normalization:** Standardize the numeric field.
4. **Grouping:** Compute summary statistics grouped by another field if available.

- Make sure to use the field and record set `@id`s for all references.

In [ ]:
# === Example EDA on the first available record set and its numeric field ===

from pandas.api.types import is_numeric_dtype

if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    print(f"Analyzing record set: {rs_id}")
    
    # Try to find a numeric column (typically coefficient, log likelihood, or similar field)
    numeric_field_id = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    
    if numeric_field_id is not None:
        print(f"Using numeric field (by @id): {numeric_field_id}")
        
        # Filter: Keep records where value > threshold (e.g., median as threshold for demo)
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        
        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try grouping by another (categorical) field if available
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < len(df) and not is_numeric_dtype(df[col]):
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std'])
            print(f"\nGrouped filtered data by {group_field_id}:")
            display(grouped_df)
        else:
            print("\nNo suitable categorical group field found.")
    else:
        print("No numeric field found in this record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize a numeric data distribution or relationships. For instance, display a histogram or boxplot of the key numeric field, grouped by a categorical variable if available.

- All references must use field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=32, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    if group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(y=numeric_field_id, x=group_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.show()

## 6. Conclusion

- We have loaded and explored the FAIR² dataset using its Croissant schema and the `mlcroissant` library.
- Record sets and fields were referenced by their `@id`, ensuring schema consistency and reusability.
- Through simple EDA and visualization, we demonstrated the value of standardized metadata-driven data exploration workflows.

For deeper analysis or machine learning, continue exploring fields of interest by `@id`, and consult each field's semantic documentation in the Croissant schema.